# 03 · Swahili → English translation benchmark on GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/claverfred/real_time_swahili_english_interpreter/blob/main/notebooks/03_mt_benchmark.ipynb)

Compares translation candidates on **FLORES-200 devtest** (1012 sentences, independent professional
human translations, never model output). Each candidate runs in its own process with `fallback.policy=fail`.

Reported: BLEU / chrF / chrF++ with 95% bootstrap CIs (sacrebleu signatures in each report), digits kept,
names vs reference, omission/addition proxies, latency p50/p95/p99 per sentence, peak GPU memory.

Licenses matter: every NLLB checkpoint is **CC-BY-NC-4.0 (non-commercial)**; Marian opus-mt is Apache-2.0.
The CPU reference (development PC): NLLB-600M int8, beam 2, BLEU 39.0, chrF 61.6.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100/L40S are faster). Optional: add a Hugging Face token as the Colab Secret `HF_TOKEN` (key icon on the left); every default model is public, so it is not required.

Results, converted models and data are kept on Google Drive in `MyDrive/swahili_interpreter/`, so later sessions start faster. Nothing is uploaded anywhere else.

In [ ]:
# @title 1. Setup: clone/update the repository, mount Drive, install, fetch data (safe to re-run)
REPO_URL = "https://github.com/claverfred/real_time_swahili_english_interpreter.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
PROFILE = "colab"  # @param {type:"string"}

import os, subprocess, sys
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(WORKDIR, REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git"))
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %cd  %s"], capture_output=True, text=True).stdout)

from src.platform import colab
info = colab.bootstrap(REPO, use_drive=USE_DRIVE)
print("persistent root:", info["root"], "(Google Drive)" if info["on_drive"] else "")
for n in info["notes"]:
    print("NOTE:", n)
print("Hugging Face token:", "from Colab Secrets" if info["hf_login"] else "none (fine: all default models are public)")

r = subprocess.run([sys.executable, "scripts/setup_colab.py", "--profile", PROFILE, "--no-smoke"])
stale = colab.restart_required()
if stale:
    print("Restart the runtime (Runtime > Restart session), then run this cell again:", stale)
assert r.returncode == 0, "setup reported a failure (see the table above)"

In [ ]:
# @title 2. Candidates
N = 1012  # @param {type:"integer"}
CANDIDATES = [  # (label, backend, model, beam)
    ("NLLB-600M distilled, beam 4 (colab default)", "nllb", "facebook/nllb-200-distilled-600M", 4),
    ("NLLB-600M distilled, beam 2 (cpu default)", "nllb", "facebook/nllb-200-distilled-600M", 2),
    ("NLLB-1.3B distilled, beam 4", "nllb", "facebook/nllb-200-distilled-1.3B", 4),
    ("NLLB-1.3B, beam 4", "nllb", "facebook/nllb-200-1.3B", 4),
    ("NLLB-3.3B, beam 4", "nllb", "facebook/nllb-200-3.3B", 4),
    ("Marian opus-mt-swc-en, beam 4", "marian", "Helsinki-NLP/opus-mt-swc-en", 4),
]

In [ ]:
# @title 3. Run (one process per candidate)
runs = []
for label, backend, model, beam in CANDIDATES:
    print(f"\n===== {label} =====")
    r = colab.run_cli(["evaluate", "mt", "--n", str(N), "--profile", PROFILE, "--set", f"mt.backend={backend}",
                       "--set", f"mt.model={model}", "--set", "mt.revision=main", "--set", f"mt.beam_size={beam}",
                       "--set", "fallback.policy=fail"], prefix="eval-mt")
    runs.append({"label": label, **r})
    print(f"-> exit {r['returncode']}, {r['seconds']} s, peak GPU memory {r['gpu_peak_mb']} MB")

In [ ]:
# @title 4. Comparison table (saved as CSV)
import pandas as pd
from IPython.display import display
from src.eval.summary import summarize_run

def table(runs, extra_keys=("label", "gpu_peak_mb", "seconds")):
    rows = []
    for r in runs:
        if not r.get("run_dir"):
            rows.append({"label": r.get("label"), "status": f"FAILED (exit {r['returncode']})"})
            continue
        for row in summarize_run(r["run_dir"]):
            rows.append({**{k: r.get(k) for k in extra_keys}, **row, "status": "ok"})
    return pd.DataFrame(rows)

GPU = (colab.environment_report()["cuda"]["gpus"] or [{"name": "cpu"}])[0]["name"].replace(" ", "_")

df = table(runs)
cols = ["label", "status", "n", "bleu", "bleu_ci95", "chrf", "chrf_ci95", "chrf_pp", "numbers_kept", "names_vs_ref",
        "omission_rate", "latency_p50_s", "latency_p95_s", "latency_p99_s", "gpu_peak_mb", "mt_model"]
display(df[[c for c in cols if c in df]])
csv = colab.results_dir() / f"colab_mt_comparison_{GPU}.csv"
df.to_csv(csv, index=False)
print("saved", csv)

In [ ]:
# @title 5. Chart: quality vs latency
import matplotlib.pyplot as plt
ok = df[df.status == "ok"]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(ok.latency_p50_s, ok.chrf, s=60, color="#2a78d6")
for _, r in ok.iterrows():
    ax.annotate(r.label, (r.latency_p50_s, r.chrf), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel("median latency per sentence (s)"); ax.set_ylabel("chrF (higher is better)")
ax.set_title(f"FLORES-200 devtest, n={N}, {GPU}"); ax.grid(color="#e1e0d9")
png = colab.results_dir() / f"colab_mt_comparison_{GPU}.png"
fig.savefig(png, dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
RUN_DIRS, EXTRA_FILES, EXPORT_NAME = [r["run_dir"] for r in runs], [csv, png], "mt-benchmark"
# @title Export: ZIP this notebook's runs (JSON, CSV, audio, logs, images, manifests) and download
import time
zip_path = colab.results_dir() / f"{EXPORT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}.zip"
env_files = sorted((colab.results_dir().parent / "env").glob("*"))[-4:]
z = colab.zip_runs([p for p in RUN_DIRS if p], zip_path, extra_files=EXTRA_FILES + env_files)
print(z)
colab.download(zip_path)